# TTA Param Sweep: DOTA / TDA / GDA / ETTA

Notebook này chạy param sweep cho nhóm method mới đã được implement trong repo:

- `DOTA` -> `dota`
- `TDA` -> `tda`
- `GDA` -> `gda`
- `ETTA` -> `etta`

Các method là feature-level adapters tương thích pipeline embeddings/probe hiện tại.


## Kaggle Setup


In [ ]:
!git clone -b dev https://github.com/hoavien0110/training-free-tta-for-deepfake-detection.git /kaggle/working/training-free-tta-for-deepfake-detection
%cd /kaggle/working/training-free-tta-for-deepfake-detection
!pip install -q -e . --no-deps


## Check Inputs


In [ ]:
!ls -lah /kaggle/input
!find /kaggle/input -maxdepth 3 -type f \( -name "*.pt" -o -name "*.csv" \) | sort | sed -n "1,180p"


## Method Availability


In [ ]:
from deepfake_tta.methods import AVAILABLE_TTA_METHODS

REQUESTED_METHODS = {
    "DOTA": "dota",
    "TDA": "tda",
    "GDA": "gda",
    "ETTA": "etta",
}

available = set(AVAILABLE_TTA_METHODS)
available_requested = {name: method for name, method in REQUESTED_METHODS.items() if method in available}
missing_requested = {name: method for name, method in REQUESTED_METHODS.items() if method not in available}

print("Available requested methods:", available_requested)
print("Missing requested methods:", missing_requested)
print("All registry methods:", AVAILABLE_TTA_METHODS)


## Run Param Sweep

Cell này sweep `none + dota + tda + gda + etta`. Có thể giảm grid nếu Kaggle chạy lâu.


In [ ]:
FFPP_SPLIT = "/kaggle/input/ffpp-split-features"
FFPP_CORR = "/kaggle/input/ffpp-test-embeddings/ffpp_test_embeddings"
CELEB_CORR = "/kaggle/input/deepfakebench-features"
MODELS = "/kaggle/input/ffpp-training-free-models"
MODELS_BAL = "/kaggle/input/ffpp-training-free-models-balanced"

METHODS_TO_SWEEP = ["none"] + [method for method in REQUESTED_METHODS.values() if method in available]

print("Sweeping:", METHODS_TO_SWEEP)
assert len(METHODS_TO_SWEEP) > 1, "No requested implemented methods are available for param sweep yet."


## Block Size

Dùng cho balanced-aligned corruption datasets.


In [ ]:
BLOCK_SIZE = 16


In [ ]:
METHOD_ARGS = " ".join(METHODS_TO_SWEEP)

!python testing/evaluate_tta_param_sweep.py \
  --train-features {FFPP_SPLIT}/ffpp_train_features.pt \
  --dataset ffpp-test={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption={FFPP_CORR} \
  --dataset celebdfv1-test-corruption={CELEB_CORR} \
  --dataset ffpp-test-balanced={FFPP_SPLIT}/ffpp_test_features.pt \
  --dataset ffpp-test-corruption-balanced={FFPP_CORR} \
  --dataset celebdfv1-test-corruption-balanced={CELEB_CORR} \
  --balanced-dataset ffpp-test-balanced \
  --balanced-aligned-dataset ffpp-test-corruption-balanced \
  --balanced-aligned-dataset celebdfv1-test-corruption-balanced \
  --block-size {BLOCK_SIZE} \
  --model linear-probe={MODELS}/ffpp_linear_probe_split.pt \
  --model osd={MODELS}/ffpp_osd_linear_probe_split.pt \
  --model linear-probe-balanced={MODELS_BAL}/ffpp_linear_probe_split.pt \
  --model osd-balanced={MODELS_BAL}/ffpp_osd_linear_probe_split.pt \
  --thresholds-csv linear-probe={MODELS}/thresholds.csv \
  --thresholds-csv osd={MODELS}/thresholds.csv \
  --thresholds-csv linear-probe-balanced={MODELS_BAL}/thresholds.csv \
  --thresholds-csv osd-balanced={MODELS_BAL}/thresholds.csv \
  --methods {METHOD_ARGS} \
  --dota-base-weight 0.35 0.55 0.75 \
  --dota-momentum 0.95 0.97 0.99 \
  --dota-confidence-threshold 0.0 0.8 0.9 \
  --tda-positive-alpha 0.25 0.4 \
  --tda-positive-entropy-threshold 0.25 0.4 \
  --tda-negative-alpha 0.0 0.15 \
  --gda-alpha 0.5 1.0 1.5 \
  --gda-shrinkage 0.0 0.1 0.3 \
  --etta-alpha 0.25 0.45 0.65 \
  --etta-beta 8.0 12.0 \
  --etta-momentum 0.95 0.97 \
  --etta-confidence-threshold 0.8 0.9 \
  --balance-method-fit \
  --continue-on-error \
  --results-output /kaggle/working/tta_param_sweep_dota_tda_gda_etta_results.csv


## Preview Results


In [ ]:
import pandas as pd

results = pd.read_csv("/kaggle/working/tta_param_sweep_dota_tda_gda_etta_results.csv")
display(results.head())
display(results.sort_values(["dataset", "method", "f1"], ascending=[True, True, False]).head(40))


## Best Params


In [ ]:
metric = "f1"
valid = results[results["error"].isna()] if "error" in results.columns else results
metric_cols = ["acc", "f1", "auc", "ap", "eer"]
summary = valid.groupby(["dataset", "model", "method", "param_id"], dropna=False)[metric_cols].mean().reset_index()
best = (summary.sort_values(["dataset", "model", "method", metric], ascending=[True, True, True, False])
        .groupby(["dataset", "model", "method"], as_index=False)
        .head(3))
summary.to_csv("/kaggle/working/tta_param_sweep_dota_tda_gda_etta_summary.csv", index=False)
best.to_csv("/kaggle/working/tta_param_sweep_dota_tda_gda_etta_best.csv", index=False)
display(best)


## Notes

Các implementation này chạy trên saved image embeddings, nên phần CLIP text-prompt/adaptive prompt của paper được thay bằng source class prototypes hoặc probe probabilities để phù hợp pipeline hiện tại.
